# 03 — Load a 3D Revision

Companion to [Chapter 09](../09-3d.md). This project is DMS-only, so the 3D
model/revision/node APIs are called as **raw HTTP** through `client.get` /
`client.post` rather than typed SDK methods -- good practice for the Document Parser
API in the next chapter, which is raw HTTP for a different reason (it's internal).

Cell order: auth -> find/create model shell -> find/create revision -> poll with a
timeout -> inspect nodes -> map a couple of names to assets -> bridge to the Function.

In [ ]:
# ---------------------------------------------------------------- setup ----
import os
from pathlib import Path

from cognite.client import CogniteClient, global_config
global_config.disable_pypi_version_check = True
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials, OAuthInteractive
import time

# Find the repo root by its markers, so this cell works wherever Jupyter started.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents]
             if (p / "pyproject.toml").exists() and (p / "training").exists()), HERE)

env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#") or "=" not in s:
            continue
        k, v = s.split("=", 1)
        if " #" in v and not v.startswith(('"', "'")):
            v = v.split(" #", 1)[0].rstrip()
        os.environ.setdefault(k, v)      # a real environment variable always wins

missing = [k for k in ("CDF_PROJECT", "CDF_CLUSTER", "IDP_CLIENT_ID")
           if not os.environ.get(k)]
assert not missing, f"Missing {missing}. Copy .env.example to {env_path} and fill it in."


def cdf_client(name: str) -> CogniteClient:
    """Build the client EXPLICITLY.

    `CogniteClient()` with no arguments does not read your .env. The SDK removed
    implicit construction in v8 and raises:
        ValueError: No ClientConfig has been provided
    The branch below is the two-identity rule from Chapter 02, in code.
    """
    base_url = os.environ.get("CDF_URL") or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
    scopes = [s for s in os.environ.get("IDP_SCOPES", f"{base_url}/.default").split(",") if s]

    if os.environ.get("LOGIN_FLOW", "interactive").lower() == "interactive":
        creds = OAuthInteractive(              # you, in a browser -- needs
            authority_url=os.environ["IDP_AUTHORITY_URL"],   # localhost:53000
            client_id=os.environ["IDP_CLIENT_ID"],           # as a redirect URI
            scopes=scopes)
    else:
        creds = OAuthClientCredentials(        # unattended: a service principal
            token_url=os.environ["IDP_TOKEN_URL"],
            client_id=os.environ["IDP_CLIENT_ID"],
            client_secret=os.environ["IDP_CLIENT_SECRET"],
            scopes=scopes)

    return CogniteClient(ClientConfig(
        client_name=name, project=os.environ["CDF_PROJECT"],
        base_url=base_url, credentials=creds))


YOURNAME = os.environ.get("PARTICIPANT", "YOURNAME")   # [CHANGE] if not in .env
client   = cdf_client(f"dm-handson-{YOURNAME}-load-3d")

space       = f"isp_{YOURNAME}_TRN"
schema_edm  = f"ssp_{YOURNAME}_TrainingCore_edm"
schema_sdm  = f"ssp_{YOURNAME}_MaintenanceInsight_sdm"
raw_db      = f"rwd_{YOURNAME}_Training_TRN"
model_version = "v1.0.0"


# --- identifiers every chapter uses ---------------------------------------
from cognite.client.data_classes.data_modeling import ViewId
from cognite.client.data_classes import filters as flt
from cognite.client.data_classes.data_modeling.query import (
    Query, QuerySync, NodeResultSetExpression, EdgeResultSetExpression,
    Select, SourceSelector)
from cognite.client.data_classes.data_modeling import (
    NodeId, EdgeId, NodeApply, EdgeApply, NodeOrEdgeData, DirectRelationReference)
from cognite.client.data_classes.raw import Row

from cognite.client.data_classes.aggregations import Count, Avg, Max

INSTANCE_SPACE = space
EDM_SPACE      = schema_edm
SDM_SPACE      = schema_sdm
RAW_DB         = raw_db
MODEL_VERSION  = model_version

ASSET      = ViewId("cdf_cdm", "CogniteAsset",     "v1")
EQUIPMENT  = ViewId("cdf_cdm", "CogniteEquipment", "v1")
ACTIVITY   = ViewId("cdf_cdm", "CogniteActivity",  "v1")
TIMESERIES = ViewId("cdf_cdm", "CogniteTimeSeries","v1")
FILE       = ViewId("cdf_cdm", "CogniteFile",      "v1")
WORKORDER  = ViewId(EDM_SPACE, "WorkOrder",              MODEL_VERSION)
EHP        = ViewId(SDM_SPACE, "EquipmentHealthProfile", MODEL_VERSION)


file_xid   = f"file_{YOURNAME}_TRN_3D_21_SEP"
model_name = f"trd_{YOURNAME}_TRN_CAD"
project    = client.config.project

print("connected:", client.config.project, "| space:", space)

## Step 1 -- find or create the 3D model shell

On a DMS-only project the model shell needs `space` + `type: CAD` -- fields the
classic 3D loaders don't expect, which is why this isn't a typed SDK call.

In [ ]:
base_models = f"/api/v1/projects/{project}/3d/models"
payload = client.get(base_models, params={"limit": 1000}).json()
model = next((m for m in payload.get("items", []) if m.get("name") == model_name), None)

if model is None:
    # The /3d/models create payload is PROJECT-DEPENDENT, and there is no flag to
    # ask about it first:
    #   * a DMS-enabled 3D project REQUIRES {"name", "space", "type"}
    #   * a classic 3D project REJECTS space, with "space is not supported"
    # Try the DMS shape, fall back to classic.
    for item in ({"name": model_name, "space": space, "type": "CAD"},
                 {"name": model_name}):
        try:
            model = client.post(base_models, json={"items": [item]}).json()["items"][0]
            break
        except Exception as exc:
            if "space is not supported" not in str(exc):
                raise
    if model is None:
        raise RuntimeError(f"could not create 3D model {model_name!r}")

model_id = model["id"]
flavour = "dms" if model.get("space") else "classic"
print("model id:", model_id, "name:", model["name"], "| 3D API flavour:", flavour)

## Step 2 -- find or create a revision from the classic OBJ file

In [ ]:
base_revisions = f"{base_models}/{model_id}/revisions"
revisions = client.get(base_revisions, params={"limit": 100}).json().get("items", [])

if revisions:
    revision = revisions[0]
    print("found existing revision:", revision["id"], "status:", revision.get("status"))
else:
    src = client.files.retrieve(external_id=file_xid)
    assert src is not None and src.uploaded, f"OBJ classic file {file_xid} missing or not uploaded"
    session = client.iam.sessions.create()
    resp = client.post(base_revisions, json={"items": [{"fileId": src.id, "published": True, "nonce": session.nonce}]})
    revision = resp.json()["items"][0]
    print("created revision:", revision["id"])

revision_id = revision["id"]

## Step 3 -- poll with a timeout (`Queued` / `Processing` -> `Done` | `Failed`)

Conversion can take minutes on a real model. This cell has a bounded budget and will
hand control back rather than block forever -- rerun the cell later if it times out.

In [ ]:
deadline = time.time() + 20 * 60
status = revision.get("status")
while status in ("Queued", "Processing") and time.time() < deadline:
    time.sleep(15)
    revision = client.get(f"{base_revisions}/{revision_id}").json()
    status = revision.get("status")
    print("status:", status)

print("final status:", status)
if status not in ("Done",):
    print("Not done yet (or Failed) -- rerun this cell later rather than creating a second revision.")

## Step 4 -- publish (if `Done` and not already published), then list nodes

In [ ]:
if status == "Done" and not revision.get("published"):
    body = {"items": [{
        "id": revision_id,
        "instanceId": {"space": space, "externalId": f"cog_3d_revision_{revision_id}"},
        "update": {"published": {"set": True}},
    }]}
    resp = client.post(f"{base_revisions}/update", json=body)
    revision = resp.json()["items"][0]
    print("published:", revision.get("published"))

nodes = []
if status == "Done":
    nodes = client.get(f"{base_revisions}/{revision_id}/nodes", params={"limit": 1000}).json().get("items", [])
    print(f"{len(nodes)} nodes; example names: {sorted({n.get('name') for n in nodes if n.get('name')})[:10]}")

## Step 5 -- inspect one mapping by hand (pump A)

The full Function maps five tags plus the deck; here, just look at one node to see
the shape of what you're working with before the Function does all of them.

In [ ]:
pump_node = next((n for n in nodes if n.get("name") == "21-PA-2001A"), None)
print("pump node:", pump_node)

## Bridge to the Function

Package this into `Load3DRevision`: the full `TAG_MAP` for all five tags plus the
deck, the `Cognite3DObject`/`CogniteCADNode`/`Cognite3DModel`/`Cognite3DRevision`/
`CogniteCADRevision` node writes that make the asset's 3D tab actually render, and
the same resume-not-restart discipline (checking for an existing revision before
creating a new one) -- but as something you can safely call repeatedly from a
workflow, not something you babysit interactively. See
[Chapter 09, section 9.4](../09-3d.md#94-write-the-function-load3drevision).